# NB3 — Ablation Part 2 (α=0.8 | no-R-matrix) + Baselines (DenseNet121 | ViT-B/16)

GPU time: ~75-90 min. Run AFTER NB2. Trains 4 models. Saves logits + result JSONs to Drive.

**Before running:** Connect T4 GPU (Runtime → Change runtime type → T4 GPU)

**Drive folder:** `MyDrive/CNN_GNN_Results/` must exist (created automatically on first run).

In [ ]:
import os, hashlib, random, warnings, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             f1_score, accuracy_score)
from scipy.stats import chi2
import warnings; warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

CLASS_NAMES  = ['Cardiomegaly','Covid-19','Normal',
                'Pneumonia','Pneumothorax','Tuberculosis']
NUM_CLASSES  = 6
DATASET_PATH = '/content/drive/MyDrive/Dataset'
RESULTS_DIR  = '/content/drive/MyDrive/CNN_GNN_Results'
CKPT_PATH    = f'{RESULTS_DIR}/cnn_gnn_final.pth'
BATCH_SIZE   = 32
IMAGE_SIZE   = 224
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')


Mounted at /content/drive
Device: cuda
Setup complete.


In [ ]:
# ── EXACT model from Untitled7.ipynb ─────────────────────────
class FeatureBasedGNN(nn.Module):
    def __init__(self, num_classes=6, feature_dim=256, hidden_dim=128):
        super().__init__()
        self.disease_prototypes = nn.Parameter(
            torch.randn(num_classes, hidden_dim) * 0.01)
        self.relation_matrix = nn.Parameter(
            torch.randn(num_classes, num_classes) * 0.01)
        self.image_projector = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2))
        self._adj = None

    def forward(self, feat):
        img_feat   = self.image_projector(feat)
        sim        = torch.matmul(img_feat, self.disease_prototypes.t())
        proto_sim  = torch.matmul(self.disease_prototypes,
                                  self.disease_prototypes.t())
        adj        = torch.sigmoid(self.relation_matrix + 0.1 * proto_sim)
        adj_norm   = adj / (adj.sum(dim=1, keepdim=True) + 1e-8)
        self._adj  = adj_norm.detach().cpu()
        propagated = torch.matmul(sim, adj_norm)
        return 0.7 * sim + 0.3 * propagated   # raw logits

    def get_relationship_matrix(self):
        if self._adj is not None: return self._adj.numpy()
        with torch.no_grad():
            proto_sim = torch.matmul(self.disease_prototypes,
                                     self.disease_prototypes.t())
            adj = torch.sigmoid(self.relation_matrix + 0.1 * proto_sim)
            adj_norm = adj / (adj.sum(dim=1, keepdim=True) + 1e-8)
        return adj_norm.cpu().numpy()


class CompleteMedicalModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        mobilenet    = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        for i, block in enumerate(self.encoder):
            for p in block.parameters():
                p.requires_grad = (i >= 15)
        self.feature_proj = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.2))
        self.gnn = FeatureBasedGNN(num_classes, 256, 128)

    def forward(self, x):
        feat = self.encoder(x)
        feat = self.gap(feat).flatten(1)
        feat = self.feature_proj(feat)
        return self.gnn(feat)   # raw logits

print('Proposed model class ready.')


Proposed model class ready.


In [ ]:
# ── Ablation model variants ───────────────────────────────────

# A0: CNN-only — same MobileNetV2 encoder, plain head, no GNN
class CNNOnlyModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        mobilenet    = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        for i, block in enumerate(self.encoder):
            for p in block.parameters():
                p.requires_grad = (i >= 15)
        self.head = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.head(self.gap(self.encoder(x)).flatten(1))


# A1/A2/A4: GNN with different alpha (direct vs propagated weighting)
class GNNAlpha(nn.Module):
    def __init__(self, num_classes=6, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        mobilenet    = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        for i, block in enumerate(self.encoder):
            for p in block.parameters():
                p.requires_grad = (i >= 15)
        self.feature_proj = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2))
        self.disease_prototypes = nn.Parameter(
            torch.randn(num_classes, 128) * 0.01)
        self.relation_matrix = nn.Parameter(
            torch.randn(num_classes, num_classes) * 0.01)
        self.projector = nn.Sequential(
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2))

    def forward(self, x):
        feat       = self.gap(self.encoder(x)).flatten(1)
        feat       = self.feature_proj(feat)
        img_feat   = self.projector(feat)
        sim        = torch.matmul(img_feat, self.disease_prototypes.t())
        proto_sim  = torch.matmul(self.disease_prototypes,
                                  self.disease_prototypes.t())
        adj        = torch.sigmoid(self.relation_matrix + 0.1 * proto_sim)
        adj_norm   = adj / (adj.sum(dim=1, keepdim=True) + 1e-8)
        propagated = torch.matmul(sim, adj_norm)
        return self.alpha * sim + (1 - self.alpha) * propagated


# A5: GNN without learnable relation matrix — only prototype similarity
class GNNNoRelationMatrix(GNNAlpha):
    def __init__(self, num_classes=6):
        super().__init__(num_classes, alpha=0.7)
        self.relation_matrix.data.zero_()
        self.relation_matrix.requires_grad = False

print('Ablation model classes ready.')


Ablation model classes ready.


In [ ]:
# ── New baseline models (required by reviewers) ───────────────

# DenseNet121 — CheXNet-style (Rajpurkar et al., 2017)
class DenseNet121Baseline(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        dn = models.densenet121(pretrained=True)
        self.features = dn.features
        self.gap      = nn.AdaptiveAvgPool2d((1, 1))
        # Freeze all; unfreeze last dense block + norm
        for p in self.features.parameters(): p.requires_grad = False
        for p in self.features.denseblock4.parameters(): p.requires_grad = True
        for p in self.features.norm5.parameters():       p.requires_grad = True
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(1024, num_classes))
    def forward(self, x):
        f = F.relu(self.features(x), inplace=True)
        return self.classifier(self.gap(f).flatten(1))


# ViT-B/16 — Vision Transformer baseline
class ViTBaseline(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.vit = models.vit_b_16(pretrained=True)
        # Freeze all; unfreeze last 2 transformer blocks + head
        for p in self.vit.parameters(): p.requires_grad = False
        for p in self.vit.encoder.layers[-2:].parameters(): p.requires_grad = True
        hidden = self.vit.heads.head.in_features
        self.vit.heads = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(hidden, num_classes))
    def forward(self, x): return self.vit(x)

print('Baseline model classes ready.')


Baseline model classes ready.


In [ ]:
# ── Dataset (same transforms as Untitled7) ───────────────────
class ChestXrayDataset(Dataset):
    def __init__(self, root_dir, transform, split):
        self.transform   = transform
        self.image_paths = []
        self.labels      = []
        self.filename_hashes = []   # for filename-collision sanity check (NOT a patient-ID check)
        split_path = os.path.join(root_dir, split)
        print(f'Loading [{split}]...')
        for idx, cls in enumerate(CLASS_NAMES):
            d = os.path.join(split_path, cls)
            if not os.path.exists(d): continue
            files = [f for f in os.listdir(d)
                     if f.lower().endswith(('.png','.jpg','.jpeg'))]
            for f in files:
                self.image_paths.append(os.path.join(d, f))
                self.labels.append(idx)
                self.filename_hashes.append(hashlib.md5(f.encode()).hexdigest()[:8])
            print(f'  {cls:<16}: {len(files)}')
        print(f'  Total: {len(self.image_paths)}')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, i):
        try: return self.transform(
            Image.open(self.image_paths[i]).convert('RGB')), self.labels[i]
        except: return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE), 0


train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = ChestXrayDataset(DATASET_PATH, train_tf, 'train')
val_ds   = ChestXrayDataset(DATASET_PATH, val_tf,   'valid')
test_ds  = ChestXrayDataset(DATASET_PATH, val_tf,   'test')
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Filename-collision sanity check across splits.
# IMPORTANT: this hashes the FILENAME STRING only, not file content and not a
# real patient identifier. A nonzero count here typically means class folders
# reuse generic filenames (e.g. "1.png") across splits and is NOT evidence of
# image or patient leakage. The actual leakage safeguard used in the paper is
# the separate full-file MD5 content hash reported in the Limitations section.
tr_p = set(train_ds.filename_hashes)
va_p = set(val_ds.filename_hashes)
te_p = set(test_ds.filename_hashes)
print(f'Filename-collision check (informational only, NOT a leakage check) — '
      f'Train∩Val:{len(tr_p&va_p)}  Train∩Test:{len(tr_p&te_p)}  Val∩Test:{len(va_p&te_p)}')
print('Nonzero counts are expected if filenames repeat across class folders '
      'and do not indicate patient or image leakage on their own.')
print('Dataset ready.')


In [ ]:
# ── Loss, metrics, training helpers ─────────────────────────
class MultiClassFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def get_metrics(logits_t, tgts_t):
    probs = F.softmax(logits_t, dim=1)
    preds = probs.argmax(1).numpy(); t = tgts_t.numpy()
    acc  = accuracy_score(t, preds)
    mf1  = f1_score(t, preds, average='macro', zero_division=0)
    pf1  = f1_score(t, preds, average=None,
                    labels=list(range(NUM_CLASSES)), zero_division=0)
    cm   = confusion_matrix(t, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for c in range(NUM_CLASSES):
        TP=cm[c,c]; FN=cm[c,:].sum()-TP; FP=cm[:,c].sum()-TP; TN=cm.sum()-TP-FN-FP
        sens.append(TP/(TP+FN+1e-8)); spec.append(TN/(TN+FP+1e-8))
    oh = F.one_hot(tgts_t, NUM_CLASSES).numpy(); auc = []
    for c in range(NUM_CLASSES):
        try: auc.append(roc_auc_score(oh[:,c], probs[:,c].numpy()))
        except: auc.append(float('nan'))
    return {'acc':acc,'mf1':mf1,'pf1':pf1,
            'sens':np.array(sens),'spec':np.array(spec),
            'auc':auc,'cm':cm,'probs':probs}


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); all_logits, all_tgts = [], []
    for imgs, lbls in loader:
        all_logits.append(model(imgs.to(device)).cpu())
        all_tgts.append(lbls)
    logits = torch.cat(all_logits); tgts = torch.cat(all_tgts)
    return logits, tgts, get_metrics(logits, tgts)


def print_test_results(m, label='TEST'):
    print(f'\n{"="*55}\n{label}\n{"="*55}')
    print(f'Accuracy : {m["acc"]*100:.2f}%')
    print(f'Macro F1 : {m["mf1"]:.4f}')
    print(f'Mean AUC : {np.nanmean(m["auc"]):.4f}')
    print(f'Mean Sens: {m["sens"].mean():.4f}')
    print(f'Mean Spec: {m["spec"].mean():.4f}')
    print(f'\n{"Disease":<16} {"F1":>7} {"AUC":>7} {"Sens":>7} {"Spec":>7}')
    print('-'*46)
    for i, name in enumerate(CLASS_NAMES):
        print(f'{name:<16} {m["pf1"][i]:>7.4f} {m["auc"][i]:>7.4f} '
              f'{m["sens"][i]:>7.4f} {m["spec"][i]:>7.4f}')


class EarlyStopping:
    def __init__(self, patience=10):
        self.patience=patience; self.counter=0; self.best=float('inf')
    def __call__(self, v):
        if v < self.best-1e-4: self.best=v; self.counter=0; return False
        self.counter += 1; return self.counter >= self.patience


def quick_train(model, name, epochs=30, patience=10, lr=0.0001):
    """
    Train any model variant and save best checkpoint + result JSON.
    Matches Untitled7 training setup exactly.
    """
    save_path = f'/content/{name}.pth'
    criterion = MultiClassFocalLoss(gamma=2.0)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    stopper   = EarlyStopping(patience)
    best_f1   = 0.0

    for ep in range(1, epochs+1):
        # ── Train ──
        model.train()
        for imgs, lbls in tqdm(train_loader, desc=f'  Train E{ep:02d}', leave=False):
            loss = criterion(model(imgs.to(device)), lbls.to(device))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # ── Validate ──
        val_logits, val_tgts, vm = evaluate(model, val_loader)
        val_loss = MultiClassFocalLoss()(val_logits, val_tgts).item()

        print(f'  E{ep:02d}  val_f1={vm["mf1"]:.4f}  '
              f'val_acc={vm["acc"]*100:.2f}%  val_loss={val_loss:.4f}')

        if vm['mf1'] > best_f1:
            best_f1 = vm['mf1']
            torch.save(model.state_dict(), save_path)
            print(f'       -> Best saved')

        if stopper(val_loss):
            print(f'  Early stop at epoch {ep}'); break

    # Load best weights
    model.load_state_dict(
        torch.load(save_path, map_location=device, weights_only=True))

    # Test evaluation
    logits, tgts, m = evaluate(model, test_loader)
    print_test_results(m, label=f'{name} TEST')

    # Save JSON result summary
    result = {
        'model': name,
        'acc':   float(m['acc']),
        'mf1':   float(m['mf1']),
        'auc':   float(np.nanmean(m['auc'])),
        'sens':  float(m['sens'].mean()),
        'spec':  float(m['spec'].mean()),
        'trainable_params': sum(p.numel() for p in model.parameters()
                                if p.requires_grad)
    }
    with open(f'{RESULTS_DIR}/{name}_result.json', 'w') as f:
        json.dump(result, f, indent=2)

    # Copy checkpoint to Drive
    shutil.copy(save_path, f'{RESULTS_DIR}/{name}.pth')
    print(f'  Saved {name}.pth + {name}_result.json to Drive.')
    return model, logits, tgts, m


print('All helpers ready.')


All helpers ready.


In [ ]:
# ── A4: GNN alpha=0.8 ─────────────────────────────────────────────────
print('\n' + '='*55)
print('A4: GNN alpha=0.8')
print('='*55)
abl_alpha08 = GNNAlpha(NUM_CLASSES, alpha=0.8).to(device)
tr_abl_alpha08 = sum(p.numel() for p in abl_alpha08.parameters() if p.requires_grad)
print(f'Trainable params: {tr_abl_alpha08:,}')

abl_alpha08, lg_abl_alpha08, tg_abl_alpha08, m_abl_alpha08 = quick_train(
    abl_alpha08, 'abl_alpha08', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_abl_alpha08, 'tgts': tg_abl_alpha08},
           f'{RESULTS_DIR}/abl_alpha08_logits.pth')
del abl_alpha08; torch.cuda.empty_cache()
print(f'A4: GNN alpha=0.8 done.')



A4: GNN alpha=0.8
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 125MB/s]


Trainable params: 2,348,772


  E01  val_f1=0.9302  val_acc=93.11%  val_loss=0.1789
       -> Best saved


  E02  val_f1=0.9423  val_acc=94.22%  val_loss=0.0916
       -> Best saved


...

  E26  val_f1=0.9780  val_acc=97.78%  val_loss=0.0239


  E27  val_f1=0.9831  val_acc=98.30%  val_loss=0.0175


  E28  val_f1=0.9853  val_acc=98.52%  val_loss=0.0171


  E29  val_f1=0.9816  val_acc=98.15%  val_loss=0.0195


  E30  val_f1=0.9809  val_acc=98.07%  val_loss=0.0193

abl_alpha08 TEST
Accuracy : 98.59%
Macro F1 : 0.9860
Mean AUC : 0.9996
Mean Sens: 0.9859
Mean Spec: 0.9972

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      1.0000  1.0000  1.0000  1.0000
Covid-19          0.9729  0.9991  0.9556  0.9982
Normal            0.9652  0.9983  0.9867  0.9884
Pneumonia         0.9889  1.0000  0.9867  0.9982
Pneumothorax      0.9911  0.9999  0.9867  0.9991
Tuberculosis      0.9978  1.0000  1.0000  0.9991
  Saved abl_alpha08.pth + abl_alpha08_result.json to Drive.
A4: GNN alpha=0.8 done.


In [ ]:
# ── A5: GNN no relation matrix ─────────────────────────────────────────────────
print('\n' + '='*55)
print('A5: GNN no relation matrix')
print('='*55)
abl_no_rel = GNNNoRelationMatrix(NUM_CLASSES).to(device)
tr_abl_no_rel = sum(p.numel() for p in abl_no_rel.parameters() if p.requires_grad)
print(f'Trainable params: {tr_abl_no_rel:,}')

abl_no_rel, lg_abl_no_rel, tg_abl_no_rel, m_abl_no_rel = quick_train(
    abl_no_rel, 'abl_no_rel', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_abl_no_rel, 'tgts': tg_abl_no_rel},
           f'{RESULTS_DIR}/abl_no_rel_logits.pth')
del abl_no_rel; torch.cuda.empty_cache()
print(f'A5: GNN no relation matrix done.')



A5: GNN no relation matrix
Trainable params: 2,348,736


  E01  val_f1=0.9326  val_acc=93.19%  val_loss=0.2208
       -> Best saved


  E02  val_f1=0.9112  val_acc=91.19%  val_loss=0.1268


  E03  val_f1=0.9476  val_acc=94.67%  val_loss=0.0737
       -> Best saved


  E04  val_f1=0.9727  val_acc=97.26%  val_loss=0.0453
       -> Best saved


...

  E26  val_f1=0.9860  val_acc=98.59%  val_loss=0.0198
       -> Best saved


  E27  val_f1=0.9853  val_acc=98.52%  val_loss=0.0219


  E28  val_f1=0.9831  val_acc=98.30%  val_loss=0.0208


  E29  val_f1=0.9831  val_acc=98.30%  val_loss=0.0231


  E30  val_f1=0.9830  val_acc=98.30%  val_loss=0.0220

abl_no_rel TEST
Accuracy : 98.52%
Macro F1 : 0.9853
Mean AUC : 0.9997
Mean Sens: 0.9852
Mean Spec: 0.9970

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      1.0000  1.0000  1.0000  1.0000
Covid-19          0.9751  0.9995  0.9556  0.9991
Normal            0.9593  0.9987  0.9956  0.9840
Pneumonia         0.9911  1.0000  0.9867  0.9991
Pneumothorax      0.9865  0.9999  0.9733  1.0000
Tuberculosis      1.0000  1.0000  1.0000  1.0000
  Saved abl_no_rel.pth + abl_no_rel_result.json to Drive.
A5: GNN no relation matrix done.


In [ ]:
# ── Baseline: DenseNet121 (CheXNet-style) ─────────────────────────────────────────────────
print('\n' + '='*55)
print('Baseline: DenseNet121 (CheXNet-style)')
print('='*55)
base_dn121 = DenseNet121Baseline(NUM_CLASSES).to(device)
tr_base_dn121 = sum(p.numel() for p in base_dn121.parameters() if p.requires_grad)
print(f'Trainable params: {tr_base_dn121:,}')

base_dn121, lg_base_dn121, tg_base_dn121, m_base_dn121 = quick_train(
    base_dn121, 'base_dn121', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_base_dn121, 'tgts': tg_base_dn121},
           f'{RESULTS_DIR}/base_dn121_logits.pth')
del base_dn121; torch.cuda.empty_cache()
print(f'Baseline: DenseNet121 (CheXNet-style) done.')



Baseline: DenseNet121 (CheXNet-style)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 172MB/s]


Trainable params: 2,166,278


  E01  val_f1=0.9137  val_acc=91.26%  val_loss=0.0884
       -> Best saved


  E02  val_f1=0.9553  val_acc=95.48%  val_loss=0.0482
       -> Best saved


...

  E26  val_f1=0.9714  val_acc=97.11%  val_loss=0.0352


  E27  val_f1=0.9707  val_acc=97.04%  val_loss=0.0338


  E28  val_f1=0.9700  val_acc=96.96%  val_loss=0.0352


  E29  val_f1=0.9715  val_acc=97.11%  val_loss=0.0310


  E30  val_f1=0.9715  val_acc=97.11%  val_loss=0.0371

base_dn121 TEST
Accuracy : 97.63%
Macro F1 : 0.9763
Mean AUC : 0.9993
Mean Sens: 0.9763
Mean Spec: 0.9953

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      0.9978  1.0000  0.9956  1.0000
Covid-19          0.9393  0.9974  0.8933  0.9982
Normal            0.9409  0.9987  0.9911  0.9769
Pneumonia         0.9933  1.0000  0.9956  0.9982
Pneumothorax      0.9866  0.9998  0.9822  0.9982
Tuberculosis      1.0000  1.0000  1.0000  1.0000
  Saved base_dn121.pth + base_dn121_result.json to Drive.
Baseline: DenseNet121 (CheXNet-style) done.


In [ ]:
# ── Baseline: ViT-B/16 ─────────────────────────────────────────────────
print('\n' + '='*55)
print('Baseline: ViT-B/16')
print('='*55)
base_vit = ViTBaseline(NUM_CLASSES).to(device)
tr_base_vit = sum(p.numel() for p in base_vit.parameters() if p.requires_grad)
print(f'Trainable params: {tr_base_vit:,}')

base_vit, lg_base_vit, tg_base_vit, m_base_vit = quick_train(
    base_vit, 'base_vit', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_base_vit, 'tgts': tg_base_vit},
           f'{RESULTS_DIR}/base_vit_logits.pth')
del base_vit; torch.cuda.empty_cache()
print(f'Baseline: ViT-B/16 done.')



Baseline: ViT-B/16
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 168MB/s]


Trainable params: 14,180,358


  E01  val_f1=0.9096  val_acc=90.89%  val_loss=0.1107
       -> Best saved


  E02  val_f1=0.9447  val_acc=94.44%  val_loss=0.0706
       -> Best saved


...

  E06  val_f1=0.9550  val_acc=95.48%  val_loss=0.0572
       -> Best saved


  E07  val_f1=0.9311  val_acc=93.11%  val_loss=0.0880


  E08  val_f1=0.9471  val_acc=94.67%  val_loss=0.0646


  E09  val_f1=0.9613  val_acc=96.07%  val_loss=0.0590
       -> Best saved


  Train E10:   3%|▎         | 11/338 [00:08<03:45,  1.45it/s]

In [ ]:
# ── Partial results table ─────────────────────────────────────
rows = []
for name, fname in [('A4: GNN α=0.8','abl_alpha08'),
                    ('A5: GNN no R-matrix','abl_no_rel'),
                    ('DenseNet121 (CheXNet)','base_dn121'),
                    ('ViT-B/16','base_vit')]:
    p = f'{RESULTS_DIR}/{fname}_result.json'
    if os.path.exists(p):
        with open(p) as f: d = json.load(f)
        rows.append({'Model':name,'Acc(%)':f'{d["acc"]*100:.2f}',
                     'Macro F1':f'{d["mf1"]:.4f}','Mean AUC':f'{d["auc"]:.4f}',
                     'Sens':f'{d["sens"]:.4f}','Spec':f'{d["spec"]:.4f}',
                     'Params(M)':f'{d["trainable_params"]/1e6:.2f}'})
df = pd.DataFrame(rows)
print('\nNB3 RESULTS:')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/ablation_part2_and_baselines.csv', index=False)
print('Saved ablation_part2_and_baselines.csv to Drive.')
